# Fair Comparison: Retrain vs. MCal (Unconditioned & Conditioned)

This notebook implements a proper train/test split comparison between:
1. **Retrain**: XGBoost retrained on 50%-binomial-ablated data
2. **MCal (Unconditioned)**: Linear calibrator trained on 50%-binomial-ablated vanilla predictions
3. **MCal (Conditioned)**: One linear calibrator per ablation fraction, each trained at its exact fraction

**Key difference from the original benchmarks:**
- Calibrators are **fitted on training data** and **evaluated on held-out test data**
- 80/20 stratified random split
- All methods share the same test set for fair KL divergence comparison

**KL metric**: $D_{KL}(\text{ablated\_dist} \| \text{clean\_dist})$ where distributions are over argmax predictions averaged across test samples.

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.utils import resample
import xgboost as xgb
from tqdm import tqdm

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print('Imports OK')

## 1. Data Loading and Train/Test Split

In [ ]:
# Load PhysioNet dataset
data_path = Path('balanced_physionet_dataset_0-30.csv')
df = pd.read_csv(data_path)
print(f'Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Target distribution:')
print(df['In-hospital_death'].value_counts())

# Separate features and target
TARGET = 'In-hospital_death'
X_all = df.drop(columns=[TARGET])
y_all = df[TARGET]

print(f'\nFeatures: {X_all.shape[1]}')
print(f'Feature names: {list(X_all.columns)}')

In [ ]:
# 80/20 stratified random split
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42
)

X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print(f'Train: {len(X_train)} samples')
print(f'  Class distribution: {y_train.value_counts().to_dict()}')
print(f'Test:  {len(X_test)} samples')
print(f'  Class distribution: {y_test.value_counts().to_dict()}')

n_features = X_train.shape[1]
print(f'\nNumber of features: {n_features}')

In [ ]:
# Imputer fitted ONLY on training features (clean data)
imputer = SimpleImputer(strategy='mean')
X_train_clean = imputer.fit_transform(X_train)  # fit here, never re-fit
X_test_clean  = imputer.transform(X_test)

print(f'X_train_clean shape: {X_train_clean.shape}')
print(f'X_test_clean shape:  {X_test_clean.shape}')
print('Imputer fitted on training data only.')

## 2. Ablation Functions

In [ ]:
def ablate_binomial(X_np: np.ndarray, p: float = 0.5, seed: int = None) -> np.ndarray:
    """
    Binomial ablation: each feature of each sample is independently
    set to NaN with probability p (coin flip).
    
    Used for: Retrain training, Unconditioned MCal training.
    """
    rng = np.random.default_rng(seed)
    X_abl = X_np.copy().astype(float)
    mask = rng.random(X_abl.shape) < p  # True where feature is removed
    X_abl[mask] = np.nan
    return X_abl


def ablate_fraction(X_np: np.ndarray, fraction: float, seed: int = None) -> np.ndarray:
    """
    Fraction ablation: exactly floor(n_features * fraction) features
    are randomly removed per sample (MCAR).
    
    Used for: evaluation at each ablation level, conditioned MCal training.
    """
    rng = np.random.default_rng(seed)
    X_abl = X_np.copy().astype(float)
    n_samples, n_feats = X_abl.shape
    n_remove = int(n_feats * fraction)
    if n_remove > 0:
        for i in range(n_samples):
            cols = rng.choice(n_feats, size=n_remove, replace=False)
            X_abl[i, cols] = np.nan
    return X_abl


print('Ablation functions defined.')

## 3. XGBoost Models

In [ ]:
XGB_PARAMS = dict(
    objective='binary:logistic',
    eval_metric='logloss',
    eta=0.1,
    max_depth=6,
    seed=42,
    tree_method='hist',
    enable_categorical=False,
    n_estimators=100,
)


def train_xgboost(X: np.ndarray, y: pd.Series, missing_value: float = None) -> xgb.XGBClassifier:
    params = XGB_PARAMS.copy()
    if missing_value is not None:
        model = xgb.XGBClassifier(missing=missing_value, **params)
    else:
        model = xgb.XGBClassifier(**params)
    model.fit(X, y)
    return model

In [ ]:
# --- Vanilla XGBoost (base model for MCal) ---
# Trained on CLEAN training data
print('Training Vanilla XGBoost on clean training data...')
vanilla_model = train_xgboost(X_train_clean, y_train)
vanilla_train_acc = (vanilla_model.predict(X_train_clean) == y_train).mean()
vanilla_test_acc  = (vanilla_model.predict(X_test_clean)  == y_test).mean()
print(f'  Train acc: {vanilla_train_acc:.4f},  Test acc: {vanilla_test_acc:.4f}')

In [ ]:
# --- Retrained XGBoost ---
# Trained on binomially ablated data (p=0.5), then mean-imputed before training
print('Applying binomial ablation (p=0.5) to training data for Retrain...')
X_train_abl_binom = ablate_binomial(X_train_clean, p=0.5, seed=0)
print(f'  NaN fraction: {np.isnan(X_train_abl_binom).mean():.3f}')

# Mean-impute before training (using the training imputer)
X_train_abl_imputed = imputer.transform(X_train_abl_binom)

print('Training Retrained XGBoost...')
retrain_model = train_xgboost(X_train_abl_imputed, y_train)
retrain_train_acc = (retrain_model.predict(X_train_abl_imputed) == y_train).mean()
retrain_test_clean_acc = (retrain_model.predict(X_test_clean) == y_test).mean()
print(f'  Train acc (ablated data): {retrain_train_acc:.4f}')
print(f'  Test acc  (clean data):   {retrain_test_clean_acc:.4f}')

## 4. MCal Calibrator

In [ ]:
class MCalCalibrator(nn.Module):
    """
    Simple linear MCal calibrator: R(z) = W * log(p) + b,  softmax output.
    Identical architecture to MCal_CE with head_type='linear'.
    """
    def __init__(self, num_classes: int):
        super().__init__()
        self.head = nn.Linear(num_classes, num_classes)
        self.num_classes = num_classes

    def forward(self, ablated_probs: torch.Tensor) -> torch.Tensor:
        logits = torch.log(ablated_probs.clamp(min=1e-8))
        return F.softmax(self.head(logits), dim=1)

    def fit(
        self,
        ablated_probs: torch.Tensor,   # (N, C)  — probabilities on ablated data
        target_labels: torch.Tensor,   # (N,)    — argmax of clean predictions
        max_steps: int = 3000,
        lr: float = 1e-3,
        verbose: bool = False,
    ):
        optimizer = optim.Adam(self.parameters(), lr=lr)
        pbar = tqdm(range(max_steps), leave=False) if verbose else range(max_steps)
        for step in pbar:
            optimizer.zero_grad()
            logits = self.head(torch.log(ablated_probs.clamp(min=1e-8)))
            loss = F.cross_entropy(logits, target_labels)
            if torch.isnan(loss):
                break
            loss.backward()
            nn.utils.clip_grad_norm_(self.parameters(), 1.0)
            optimizer.step()
            if verbose:
                pbar.set_description(f'step {step}  loss {loss.item():.4e}')


def probs_from_model(model: xgb.XGBClassifier, X: np.ndarray) -> torch.Tensor:
    """Get model softmax probabilities as a torch tensor (N, C)."""
    probs = model.predict_proba(X).astype(np.float32)
    return torch.from_numpy(probs)


print('MCalCalibrator defined.')

## 5. Train MCal Calibrators on Training Data

In [ ]:
# Target labels for MCal: argmax of Vanilla predictions on CLEAN training data
# (what we want the calibrator to recover)
vanilla_clean_train_probs = probs_from_model(vanilla_model, X_train_clean)
target_labels_train = vanilla_clean_train_probs.argmax(dim=1)  # (N_train,)
print(f'Target label distribution (train): {torch.bincount(target_labels_train).tolist()}')

In [ ]:
# --- Unconditioned MCal ---
# Trained on vanilla predictions on 50%-binomially-ablated training data
print('Training Unconditioned MCal...')
X_train_abl_binom_for_mcal = ablate_binomial(X_train_clean, p=0.5, seed=1)
X_train_abl_binom_imputed  = imputer.transform(X_train_abl_binom_for_mcal)

vanilla_abl_train_probs = probs_from_model(vanilla_model, X_train_abl_binom_imputed)

mcal_uncond = MCalCalibrator(num_classes=2)
mcal_uncond.fit(vanilla_abl_train_probs, target_labels_train, max_steps=3000, verbose=True)

print('  Done.')

In [ ]:
# --- Conditioned MCal ---
# One calibrator per ablation fraction p in {0.1, 0.2, ..., 0.9}
ABLATION_FRACTIONS = [round(p / 10, 1) for p in range(1, 10)]
print(f'Ablation fractions: {ABLATION_FRACTIONS}')

mcal_cond = {}  # fraction -> MCalCalibrator

for frac in ABLATION_FRACTIONS:
    print(f'\nTraining Conditioned MCal at p={frac:.1f}...')
    X_train_abl_frac = ablate_fraction(X_train_clean, fraction=frac, seed=int(frac * 100))
    X_train_abl_frac_imputed = imputer.transform(X_train_abl_frac)

    vanilla_abl_frac_probs = probs_from_model(vanilla_model, X_train_abl_frac_imputed)

    cal = MCalCalibrator(num_classes=2)
    cal.fit(vanilla_abl_frac_probs, target_labels_train, max_steps=3000, verbose=False)
    mcal_cond[frac] = cal
    print(f'  Done (p={frac:.1f}).')

print('\nAll conditioned calibrators trained.')

## 6. Evaluation on Test Set

In [ ]:
def argmax_dist(probs: torch.Tensor) -> torch.Tensor:
    """
    Convert probabilities to the mean one-hot argmax distribution.
    Returns a (C,) vector — the empirical class distribution over the test set.
    """
    n, c = probs.shape
    one_hot = torch.zeros_like(probs)
    one_hot[torch.arange(n), probs.argmax(dim=1)] = 1.0
    return one_hot.mean(dim=0)  # (C,)


def kl_divergence(q: torch.Tensor, p: torch.Tensor, eps: float = 1e-10) -> float:
    """
    D_KL(q || p)  — KL divergence of q from p.
    q = ablated distribution, p = clean reference distribution.
    """
    q = q.clamp(min=eps)
    p = p.clamp(min=eps)
    return (q * (q.log() - p.log())).sum().item()


print('Evaluation helpers defined.')

In [ ]:
# --- Clean reference distributions (each method uses its own baseline) ---

# Retrain: reference = Retrain predictions on CLEAN test data
retrain_clean_test_probs = probs_from_model(retrain_model, X_test_clean)
retrain_clean_dist = argmax_dist(retrain_clean_test_probs)
print(f'Retrain clean dist: {retrain_clean_dist.tolist()}')

# MCal (both variants): reference = Vanilla predictions on CLEAN test data (no calibration at p=0)
vanilla_clean_test_probs = probs_from_model(vanilla_model, X_test_clean)
vanilla_clean_dist = argmax_dist(vanilla_clean_test_probs)
print(f'Vanilla clean dist: {vanilla_clean_dist.tolist()}')

In [ ]:
# --- Evaluate all methods across ablation fractions ---

results = {
    'fractions':    ABLATION_FRACTIONS,
    'retrain':      [],
    'mcal_uncond':  [],
    'mcal_cond':    [],
}

for frac in ABLATION_FRACTIONS:
    # Ablate test data
    X_test_abl_frac = ablate_fraction(X_test_clean, fraction=frac, seed=int(frac * 100 + 999))
    X_test_abl_imputed = imputer.transform(X_test_abl_frac)

    # ---- Retrain ----
    retrain_abl_probs = probs_from_model(retrain_model, X_test_abl_imputed)
    retrain_abl_dist  = argmax_dist(retrain_abl_probs)
    kl_retrain = kl_divergence(retrain_abl_dist, retrain_clean_dist)

    # ---- Unconditioned MCal ----
    vanilla_abl_probs   = probs_from_model(vanilla_model, X_test_abl_imputed)
    with torch.no_grad():
        uncond_calibrated = mcal_uncond(vanilla_abl_probs)
    uncond_abl_dist = argmax_dist(uncond_calibrated)
    kl_uncond = kl_divergence(uncond_abl_dist, vanilla_clean_dist)

    # ---- Conditioned MCal ----
    with torch.no_grad():
        cond_calibrated = mcal_cond[frac](vanilla_abl_probs)
    cond_abl_dist = argmax_dist(cond_calibrated)
    kl_cond = kl_divergence(cond_abl_dist, vanilla_clean_dist)

    results['retrain'].append(kl_retrain)
    results['mcal_uncond'].append(kl_uncond)
    results['mcal_cond'].append(kl_cond)

    print(f'p={frac:.1f}:  Retrain={kl_retrain:.4e},  MCal-Uncond={kl_uncond:.4e},  MCal-Cond={kl_cond:.4e}')

print('\nEvaluation complete.')

## 7. Results Summary

In [ ]:
results_df = pd.DataFrame({
    'ablation_fraction': results['fractions'],
    'Retrain':           results['retrain'],
    'MCal (Uncond)':     results['mcal_uncond'],
    'MCal (Cond)':       results['mcal_cond'],
})
results_df = results_df.set_index('ablation_fraction')
print(results_df.to_string(float_format=lambda x: f'{x:.4e}'))

## 8. Plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

fracs = results['fractions']

ax.plot(fracs, results['retrain'],     marker='o', linewidth=2, label='Retrain',          color='tab:red')
ax.plot(fracs, results['mcal_uncond'], marker='s', linewidth=2, label='MCal (Uncond.)',   color='tab:orange')
ax.plot(fracs, results['mcal_cond'],   marker='^', linewidth=2, label='MCal (Cond.)',     color='tab:blue')

ax.set_xlabel('Ablation Fraction (fraction of features removed)', fontsize=12)
ax.set_ylabel('Missingness Bias  $D_{KL}$(ablated $\|$ clean)', fontsize=12)
ax.set_title('Fair Comparison: PhysioNet (train/test split, calibrators on train)', fontsize=12)
ax.set_yscale('log')
ax.set_xticks(fracs)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('physionet_fair_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to physionet_fair_comparison.png')

## 9. Discussion

**What this notebook fixes relative to the original benchmarks:**

| Issue | Original | This notebook |
|---|---|---|
| Train/test split for calibrator | No — fit and eval on same data | Yes — fit on train, eval on held-out test |
| Retrain training ablation | Binomial (p=0.5) | Binomial (p=0.5) (same) |
| MCal training ablation (Uncond.) | Same fraction as eval | Binomial (p=0.5) — same exposure as Retrain |
| MCal training ablation (Cond.) | Same fraction as eval | Exact fraction — conditioned calibrator |
| Clean reference | Fraction-0 of *same* data | Vanilla/Retrain predictions on *held-out* clean test |

**Expected findings:**
- If Retrain is still worse than MCal (Cond.), the gap reflects genuine calibration benefit beyond full model retraining.
- If Retrain becomes competitive with MCal (Cond.), the original results were largely due to train/test leakage.
- MCal (Uncond.) is the fairest comparison to Retrain — both trained with 50%-binomial ablation exposure.